## Six factors, one objective

The campaigns this view serves search six dimensions, not two. **A two-axis scatter of
`gap_width` against `walker_dwell` would be a lie here** -- two cells sitting on the same spot
can differ in the four factors the plot does not show, so the figure would read as noise in a
smooth space rather than as structure in a space with more axes than the page has.

So there is no 2-D map below. Each factor gets its own panel against the objective, which is
the honest projection: it shows which factors move the outcome and which do not, and it does
not pretend to show where the failures are.

**What to look for:** a panel with a visible slope is a factor that matters. A panel that is a
formless cloud is a factor this campaign found nothing in -- which for a search is also a
statement about where it chose to spend its evaluations.

In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json
import pandas as pd
import matplotlib.pyplot as plt

from robovast.common.analysis import CampaignDataError, open_campaign_store

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from the campaign's own store rather than the results index because that is where a
    SEARCH records what it scored -- the index holds per-run tables, and a search's unit of
    analysis is the cell. The store is also written as the search runs, so this works on a
    campaign that is still going or was never postprocessed.
    """
    # open_campaign_store rather than a sqlite3.connect on a path built here: it resolves the
    # campaign ROOT from data_dir, so this cell also works at a configuration node instead of
    # only at the campaign, and it is the one place that knows where the store lives.
    try:
        conn = open_campaign_store(data_dir)
    except CampaignDataError as exc:
        # Reported, not swallowed. "This campaign scored nothing" and "its record is not
        # here" are different answers and only the first is a result -- an empty frame
        # returned quietly reads as the first while meaning the second.
        print(f'[no data] {exc}')
        return pd.DataFrame()
    try:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    finally:
        conn.close()
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Six factors'

# One panel per search dimension, each against the objective. Ordered as the search space
# declares them, so two campaigns over the same space produce comparable pages.
FACTORS = ['gap_width', 'walker_dwell', 'walker_speed', 'lidar_noise',
           'inflation_radius', 'max_speed']

present = [f for f in FACTORS if f in scored.columns] if not scored.empty else []
missing = [f for f in FACTORS if f not in scored.columns] if not scored.empty else []
if missing:
    # Named rather than skipped: a campaign searching fewer factors is a different experiment,
    # and a page that quietly dropped the panels would look like the same one.
    print(f"[not searched by this campaign] {', '.join(missing)}")

if present:
    ncols = 3
    nrows = (len(present) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows), squeeze=False)
    flat = [a for row in axes for a in row]
    for ax, factor in zip(flat, present):
        ax.scatter(scored[factor], scored['robustness'], s=45, alpha=0.8,
                   edgecolor='black', linewidth=0.3)
        ax.axhline(0.0, color='#c9611e', linewidth=1)
        ax.set_xlabel(factor)
        ax.set_ylabel('robustness')
    for ax in flat[len(present):]:
        ax.axis('off')
    fig.suptitle('%s: %d cells, one panel per searched factor' % (TITLE, len(scored)))
    plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst crossing found  : {scored['robustness'].min():.3f}")
    print()

    # How far the search got, in order, is the number this campaign exists to produce -- the
    # comparison it feeds is "how much simulator did each sampler need to reach a given
    # depth", and the final value alone cannot answer that.
    ordered = scored.sort_values('batch') if 'batch' in scored else scored
    spent, best, reached = 0, 0.0, {}
    for _, row in ordered.iterrows():
        spent += int(row['n_samples'])
        best = min(best, row['robustness'])
        for depth in (-1.0, -1.3, -1.5):
            if best <= depth and depth not in reached:
                reached[depth] = spent
    print("What this campaign is FOR -- what six factors cost this sampler:")
    for depth in (-1.0, -1.3, -1.5):
        got = reached.get(depth)
        print(f"  runs to reach {depth:>5} : {got if got else 'never reached'}")
    print()
    print("  Read against the SAME sampler over two factors, not against the other sampler")
    print("  here alone. The question is how much each degrades when the space stops being")
    print("  small enough to cover by chance -- and a sampler that never reaches a depth is")
    print("  a different answer from one that reaches it late.")
